In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from dataclasses import dataclass
import copy
import PIL

from conv2dggm import Conv2dGGM
from linearggm import LinearGGM


In [2]:
class SReLU(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(channels))

    def forward(self, x):
        if x.dim() == 4:
            alpha = self.alpha.view(1, -1, 1, 1)
        elif x.dim() == 2:
            alpha = self.alpha.view(1, -1)
        else:
            raise ValueError(f"SReLU expected 2D or 4D input, got {x.shape}")

        return alpha * F.relu(x)

In [3]:
@dataclass
class Config:
    batch_size = 256
    num_workers = 8
    pin_memory = True
    persistent_workers = True
    prefetch_factors = 4
    lr = 1e-3
    weight_decay = 1e-4
    num_epochs = 100
    val_split = 500
    train_split = 50000 - val_split
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3), value="random"),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD), 
    ]
)


    
cfg = Config()

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
train_set, val_set = torch.utils.data.random_split(dataset, [cfg.train_split, cfg.val_split])

train_loader = torch.utils.data.DataLoader(
    train_set, 
    batch_size=cfg.batch_size, 
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers,
    prefetch_factor=cfg.prefetch_factors,
)


val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=cfg.batch_size
)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = torch.utils.data.DataLoader(
    test_set, 
    batch_size=256,
    shuffle=False,
    num_workers=cfg.num_workers,
)


/venv/main/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
class Network(nn.Module):
    def __init__(self, input_channels=3, num_classes=10, image_size=32, dropout=0.25, n_ratio=1.0):
        super().__init__()
        self.img_size = image_size
        # self.fcin = self.img_size**2
        self.cin = input_channels
        self.cout1 = 32
        self.cout2 = 32

        self.fcin = self.cout2 * (16**2)
        self.fcout1 = 128
        self.fcout2 = 128
        self.fcout3 = num_classes

        self.n_ratio=n_ratio

        self.block1 = nn.Sequential(
            nn.Conv2d(self.cin, self.cout1, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout1),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
            # nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            Conv2dGGM(self.cout1, self.cout2, kernel_size=3, stride=1, padding=1, N_scale=self.n_ratio),
            nn.BatchNorm2d(self.cout2),
            nn.ReLU(),
            # nn.Dropout(dropout),
        )
        self.block3 = nn.Sequential(
            LinearGGM(self.fcin, self.fcout1, N_scale=self.n_ratio),
            nn.BatchNorm1d(self.fcout1),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.block4 = nn.Sequential(
            LinearGGM(self.fcout1, self.fcout2, N_scale=self.n_ratio),
            nn.BatchNorm1d(self.fcout2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Linear(self.fcout2, self.fcout3)

    def forward(self, x):

        
        x = self.block1(x)
        x = self.block2(x)
    
        x = torch.flatten(x, start_dim=1)
    
        x = self.block3(x)
        x = self.block4(x)
        x = self.head(x)
    
        return x

    

In [5]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch, labels in loader:
            batch, labels = batch.to(device), labels.to(device)
    
            outputs = model(batch)
            loss = criterion(outputs, labels)
    
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
    
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += batch_size

    avg_loss = total_loss / total
    avg_acc = correct / total

    return avg_loss, avg_acc

In [6]:
cfg = Config()
model = Network(n_ratio=5.0).to(cfg.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

best_val_acc = 0.0
best_state_dict = None

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch, labels = batch.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device,)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f'Epoch [{epoch + 1}/{cfg.num_epochs}] '
        f'Train loss: {train_loss:.4f} '
        f'Train acc: {train_acc:.4f} '
        f'Val loss: {val_loss: .4f} '
        f'Val acc: {val_acc: .4f} '
    )

print(f'Best Validation Accuracy: {best_val_acc:.4f}')

model.load_state_dict(best_state_dict)
clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device,)
print(
    f'Clean Test Loss: {clean_test_loss:.4f} '
    f'Clean Test Acc: {clean_test_acc:.4f}'
)


Epoch [1/100] Train loss: 1.6450 Train acc: 0.4039 Val loss:  1.4541 Val acc:  0.4640 
Epoch [2/100] Train loss: 1.3749 Train acc: 0.5050 Val loss:  1.3447 Val acc:  0.5480 
Epoch [3/100] Train loss: 1.2735 Train acc: 0.5478 Val loss:  1.1620 Val acc:  0.5820 
Epoch [4/100] Train loss: 1.2044 Train acc: 0.5729 Val loss:  1.2047 Val acc:  0.6040 
Epoch [5/100] Train loss: 1.1710 Train acc: 0.5851 Val loss:  1.1507 Val acc:  0.6040 
Epoch [6/100] Train loss: 1.1360 Train acc: 0.5996 Val loss:  1.0990 Val acc:  0.6180 
Epoch [7/100] Train loss: 1.1069 Train acc: 0.6102 Val loss:  1.0261 Val acc:  0.6380 
Epoch [8/100] Train loss: 1.0795 Train acc: 0.6205 Val loss:  1.0024 Val acc:  0.6620 
Epoch [9/100] Train loss: 1.0606 Train acc: 0.6264 Val loss:  0.9831 Val acc:  0.6660 
Epoch [10/100] Train loss: 1.0374 Train acc: 0.6358 Val loss:  0.9705 Val acc:  0.6520 
Epoch [11/100] Train loss: 1.0224 Train acc: 0.6417 Val loss:  1.0258 Val acc:  0.6680 
Epoch [12/100] Train loss: 1.0112 Train a

In [7]:
def set_ggm_bit_flip_prob(model, p, base_seed=1234):
    """
    Set binary-weight flip probability for every LinearGGM layer.
    This does NOT modify full-precision weights.
    It only changes W_b during forward().
    """
    layer_idx = 0

    for name, layer in model.named_modules():
        if layer.__class__.__name__ in ["LinearGGM", "Conv2dGGM"]:
            layer.perturb_prob = float(p)
            layer.perturb_seed = int(base_seed + layer_idx)
            layer_idx += 1


def reset_ggm_bit_flips(model):
    """
    Turn off binary-weight flipping in every LinearGGM layer.
    """
    for name, layer in model.named_modules():
        if layer.__class__.__name__ == ["LinearGGM", "Conv2dGGM"]:
            layer.perturb_prob = 0.0

In [8]:
device = next(model.parameters()).device

flip_percentages = list(range(0, 51, 5))  # 0%, 5%, 10%, ..., 50%

results = []

for pct in flip_percentages:
    p = pct / 100.0

    set_ggm_bit_flip_prob(
        model=model,
        p=p,
        base_seed=1234,
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    results.append({
        "flip_percent": pct,
        "flip_prob": p,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_acc_percent": 100 * test_acc,
    })

    print(
        f"Flip {pct:2d}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {100 * test_acc:.2f}%"
    )


Flip  0% | Test loss: 0.6003 | Test acc: 79.49%
Flip  5% | Test loss: 0.6032 | Test acc: 79.48%
Flip 10% | Test loss: 0.6378 | Test acc: 78.84%
Flip 15% | Test loss: 0.7102 | Test acc: 77.12%
Flip 20% | Test loss: 0.8595 | Test acc: 73.45%
Flip 25% | Test loss: 1.0980 | Test acc: 66.73%
Flip 30% | Test loss: 1.4229 | Test acc: 53.95%
Flip 35% | Test loss: 1.7642 | Test acc: 37.83%
Flip 40% | Test loss: 2.1198 | Test acc: 20.46%
Flip 45% | Test loss: 2.3147 | Test acc: 11.99%
Flip 50% | Test loss: 2.4127 | Test acc: 9.10%


In [ ]:
cfg = Config()
model = Network(n_ratio=1.0).to(cfg.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

best_val_acc = 0.0
best_state_dict = None

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch, labels = batch.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device,)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f'Epoch [{epoch + 1}/{cfg.num_epochs}] '
        f'Train loss: {train_loss:.4f} '
        f'Train acc: {train_acc:.4f} '
        f'Val loss: {val_loss: .4f} '
        f'Val acc: {val_acc: .4f} '
    )

print(f'Best Validation Accuracy: {best_val_acc:.4f}')

model.load_state_dict(best_state_dict)
clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device,)
print(
    f'Clean Test Loss: {clean_test_loss:.4f} '
    f'Clean Test Acc: {clean_test_acc:.4f}'
)


Epoch [1/100] Train loss: 1.7702 Train acc: 0.3528 Val loss:  1.5932 Val acc:  0.4460 
Epoch [2/100] Train loss: 1.5243 Train acc: 0.4556 Val loss:  1.4485 Val acc:  0.4980 
Epoch [3/100] Train loss: 1.4493 Train acc: 0.4853 Val loss:  1.3926 Val acc:  0.5220 
Epoch [4/100] Train loss: 1.3978 Train acc: 0.5063 Val loss:  1.3359 Val acc:  0.5220 
Epoch [5/100] Train loss: 1.3602 Train acc: 0.5229 Val loss:  1.2838 Val acc:  0.5620 
Epoch [6/100] Train loss: 1.3450 Train acc: 0.5296 Val loss:  1.3136 Val acc:  0.5500 
Epoch [7/100] Train loss: 1.3195 Train acc: 0.5410 Val loss:  1.2790 Val acc:  0.5560 
Epoch [8/100] Train loss: 1.3016 Train acc: 0.5502 Val loss:  1.1866 Val acc:  0.5900 
Epoch [9/100] Train loss: 1.2956 Train acc: 0.5489 Val loss:  1.2211 Val acc:  0.5820 
Epoch [10/100] Train loss: 1.2812 Train acc: 0.5580 Val loss:  1.1930 Val acc:  0.5960 
Epoch [11/100] Train loss: 1.2669 Train acc: 0.5619 Val loss:  1.2108 Val acc:  0.6180 
Epoch [12/100] Train loss: 1.2671 Train a

In [ ]:
device = next(model.parameters()).device

flip_percentages = list(range(0, 51, 5))  # 0%, 5%, 10%, ..., 50%

results = []

for pct in flip_percentages:
    p = pct / 100.0

    set_ggm_bit_flip_prob(
        model=model,
        p=p,
        base_seed=1234,
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    results.append({
        "flip_percent": pct,
        "flip_prob": p,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_acc_percent": 100 * test_acc,
    })

    print(
        f"Flip {pct:2d}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {100 * test_acc:.2f}%"
    )


In [ ]:
cfg = Config()
model = Network(n_ratio=2.0).to(cfg.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

best_val_acc = 0.0
best_state_dict = None

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch, labels = batch.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device,)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f'Epoch [{epoch + 1}/{cfg.num_epochs}] '
        f'Train loss: {train_loss:.4f} '
        f'Train acc: {train_acc:.4f} '
        f'Val loss: {val_loss: .4f} '
        f'Val acc: {val_acc: .4f} '
    )

print(f'Best Validation Accuracy: {best_val_acc:.4f}')

model.load_state_dict(best_state_dict)
clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device,)
print(
    f'Clean Test Loss: {clean_test_loss:.4f} '
    f'Clean Test Acc: {clean_test_acc:.4f}'
)


In [ ]:
device = next(model.parameters()).device

flip_percentages = list(range(0, 51, 5))  # 0%, 5%, 10%, ..., 50%

results = []

for pct in flip_percentages:
    p = pct / 100.0

    set_ggm_bit_flip_prob(
        model=model,
        p=p,
        base_seed=1234,
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    results.append({
        "flip_percent": pct,
        "flip_prob": p,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_acc_percent": 100 * test_acc,
    })

    print(
        f"Flip {pct:2d}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {100 * test_acc:.2f}%"
    )
